# **大模型在医学中的应用入门**

## **本节课学习目标**

在本节课中，你将学到：
- 理解大模型（LLM）的基本工作原理
- 学会使用 API 调用大模型
- 掌握如何通过提示词（Prompt）控制模型输出
- 实战：用大模型从病例文本中提取结构化信息

## **医学应用场景**

想象这样一个场景：

**你是医院的一名住院医师**，每天需要处理大量的患者病历。这些病历往往是自由文本形式，包含患者的主诉、现病史、检查结果等。你想把这些杂乱的文本信息整理成结构化的表格，方便后续的数据分析和研究。

传统方法需要人工逐条阅读和录入，耗时耗力。**大模型可以帮我们自动化这个过程！**

此外，大模型还可以用于：
- 辅助诊断建议
- 医学文献总结
- 与患者沟通生成
- ……

让我们从基础开始，一步步掌握这个强大工具的使用方法。

## **大模型基本原理**

### **1. Tokenizer（分词器）——把文本变成积木块**

**问题：** 计算机不认识"糖尿病"、"高血压"这些词，它只认识数字。怎么办？

**答案：** 我们需要先把文本"切碎"成模型能认识的最小单位，这些单位叫做 **Token**。

**比喻：**
- Token 就像乐高积木块
- "糖尿病患者需要控制饮食" 这个句子可能被切分成：`["糖", "尿", "病", "患者", "需要", "控制", "饮食"]`
- 不同模型的"切法"可能不同

**为什么重要？**
- Token 是模型理解的"单词"，类似于我们学英语时的"单词"
- API 收费通常是按 Token 数量计算的

---

### **2. Embedding（嵌入）——给每个 Token 一个带语义信息的数字坐标**

**问题：** 即使把词切成了 Token，计算机还是不理解这些 Token 的含义。怎么让它理解？

**答案：** 把每个 Token 转换成一组数字，这组数字叫 **Embedding Vector（嵌入向量）**。

**比喻：**
- 想象一个巨大的"词语地图"，每个词都有一个坐标位置
- 意思相近的词，在地图上离得近
- 例如："医生"和"护士"的坐标很近，"医生"和"香蕉"的坐标很远

**为什么重要？**
- 模型通过这些数字坐标理解词语的含义和关系
- 这是模型"理解"语言的基础

---

### **3. LLM 的核心——超级强大的下一个词预测器**

**核心原理：**

大语言模型（LLM）本质上就是一个**预测下一个词是什么**的模型。通过对大量文本的训练学习，从中学习到token之间的关系，从而进行预测。由于大模型参数很大，模型表征能力非常强，因此可以对非常复杂的文本之间的关系进行表征，从而让人看起来拥有“智能”。

**比喻：**
- 想象你在玩"填空游戏"
- 题目是："糖尿病患者应该控制____"
- 你会猜："饮食"、"血糖"、"体重"……
- LLM 也是这样，它看过海量文本，知道哪个词最可能出现

**生成文本的机制：**
```
输入："糖尿病患者应该控制"
模型预测："饮食"
新输入："糖尿病患者应该控制饮食____"
模型预测："并且"
新输入："糖尿病患者应该控制饮食并且____"
模型预测："定期"
……不断重复，直到生成完整回答
```

## **实践准备：认识并使用大模型 API**

### **什么是 API 和 API Key？**

**API（应用程序接口）** —— 比喻为点餐热线

- 你（你的程序）想用大模型的能力
- 但你不需要自己造一个大模型
- 你只需要"打电话"（调用 API）给提供大模型服务的公司
- 它们帮你完成推理，然后把结果传回来

**API Key** —— 比喻为验证码或会员卡

- 每次调用 API 都需要出示 API Key
- 它证明：**"这是我，我有权使用这个服务"**
- 同时也用于计费（你用了多少token，收多少费）

---

### **如何获取 API Key？**

请访问课程文档：

获取 API Key 指南: https://my.feishu.cn/wiki/YYiGwNDO0i6jRrkQ94BcNH3SnQc

**推荐平台与模型：**
- **硅基流动**（SiliconFlow）：聚合了多个大模型
- 推荐模型：`glm-5`、`Kimi-K2.5`、`MiniMax-M2.5`、`DeepSeek-V3.2`
- 这些模型在中文问答、代码能力上表现不错

**重要提醒：**
- API Key 就像你的银行卡密码，**不要分享给别人**
- 不要把真实的 API Key 上传到 GitHub 或公开平台
- 在本 Notebook 中，我们将使用环境变量或配置文件来安全存储

---

### **常见 API 参数讲解**

调用大模型 API 时，我们可以调整一些参数来控制模型的输出：

#### **1. temperature（温度）—— 创造性/随机性控制**

- **范围**：0.0 ~ 2.0（通常使用 0.0 ~ 1.0）
- **低温度（如 0.1）**：
  - 模型更"保守"，选择概率最高的词
  - 输出更稳定、更确定
  - 适合：需要准确答案的任务（如医学知识问答）

- **高温度（如 0.8 ~ 1.0）**：
  - 模型更"冒险"，可能选择概率稍低的词
  - 输出更有创造性、更多样化
  - 适合：创意写作、头脑风暴

**比喻：**
- 低温度 = 考试时只选你最确定的答案
- 高温度 = 创意写作时尝试不同的表达方式

#### **2. top_p —— 核采样控制**

- **范围**：0.0 ~ 1.0
- **含义**：只从累积概率达到 top_p 的词中选择
- **常用值**：0.9（从概率总和为 90% 的词中选）
- **与 temperature 的关系**：通常只需要调整其中一个

**比喻：**
- top_p = 0.9 意味着"我只考虑最可能的 90% 选项，其他的太离谱的不考虑"

#### **3. max_tokens —— 最大输出长度**

- **含义**：限制模型最多生成多少个 Token
- **作用**：控制回答的长度，防止模型"喋喋不休"
- **常用值**：根据任务需要设定，如 500、1000、2000

**比喻：**
- 就像告诉对方"请用不超过 200 字回答"

## **动手实践一：初次调用与参数体验**

### **目标**

1. 安装必要的 Python 库
2. 配置 API Key
3. 编写最简单的调用函数
4. 通过调整参数，直观感受参数对输出结果的影响

### **步骤 1：安装必要的库**

我们需要安装 `openai` 库。虽然名字叫 openai，但它是一个通用的库，可以调用各种兼容 OpenAI 接口的大模型服务。

In [1]:
# 安装 openai 库（如果尚未安装）
# 在 Jupyter Notebook 中，可以在命令前加 ! 来执行 shell 命令

!pip install openai -q
!pip install dotenv

# 导入必要的库
import os
from openai import OpenAI
from dotenv import load_dotenv
import json

print("库安装和导入成功！")

库安装和导入成功！


### **步骤 2：配置 API Key**

**安全提示：**

**永远不要将真实的 API Key 直接写在代码中并提交到公开仓库！**

**推荐做法：**

1. 创建一个名为 `.env` 的文件（在项目根目录下）
2. 在文件中写入：
   ```
   OPENAI_API_KEY=你的真实API_Key
   BASE_URL=https://api.siliconflow.cn/v1
   ```
3. 使用以下代码安全地读取配置

**或者临时方法（仅用于学习）：**

在下面的代码中，我们先用一个占位符，你需要手动替换为你的真实 API Key。

In [2]:
# TODO: 请在这里填入你的真实 API Key 和 Base URL
# 提示：从你注册的平台获取这些信息

# 方法 1：直接设置（仅用于学习，不要提交到公开仓库）
# API_KEY = "YOUR API KEY"  # TODO: 替换为你的真实 API Key
# BASE_URL = "https://api.siliconflow.cn/v1"  # 硅基流动的 Base URL

# 方法 2：从环境变量读取（推荐，更安全） # TODO: 在同目录下创建.env文件并写入OPENAI_API_KEY和BASE_URL
# 如果你设置了环境变量，可以取消下面的注释
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("BASE_URL")

# 初始化客户端
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

print("API 配置完成！")
print(f"Base URL: {BASE_URL}")
print(f"API Key: {API_KEY[:10]}...{API_KEY[-4:]}")  # 只显示部分，保护隐私

API 配置完成！
Base URL: https://api.siliconflow.cn/v1
API Key: sk-xjkzypr...qsfs


### **步骤 3：编写调用函数**

现在让我们编写一个通用的函数来调用大模型。

In [3]:
# 定义一个调用大模型的通用函数

def call_llm(messages, model="Pro/zai-org/GLM-5", temperature=0.5, max_tokens=2000, top_p=0.9):
    """
    调用大语言模型
    
    参数：
        messages: 对话消息列表，格式如 [{"role": "user", "content": "..."}]
        model: 使用的模型名称，如 "glm-5", "Qwen-3.6-PLUS"
        temperature: 温度参数，控制输出的随机性（0.0-2.0）
        max_tokens: 最大输出 Token 数
        top_p: 核采样参数（0.0-1.0）
    
    返回：
        模型的回答文本
    """
    try:
        # 使用 client.chat.completions.create 调用模型，传入 messages, model, temperature, max_tokens, top_p 参数
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p
        )
        
        # 提取回答内容
        answer = response.choices[0].message.content
        return answer
        
    except Exception as e:
        return f"调用失败：{str(e)}"

print("调用函数定义完成！")

调用函数定义完成！


### **步骤 4：第一次调用——向模型提问**

让我们用一个医学问题来测试模型。

In [4]:
# 准备医学问题
medical_question = "请解释糖尿病二型的发病原理。"

# 构建消息
messages = [
    {"role": "user", "content": medical_question}
]

# 调用模型（使用默认参数）
print("正在向模型提问...")
print(f"问题：{medical_question}\n")
print("-" * 60)

# 调用 call_llm 函数
answer = call_llm(messages)

print("模型回答：")
print(answer)
print("-" * 60)

正在向模型提问...
问题：请解释糖尿病二型的发病原理。

------------------------------------------------------------
模型回答：
2型糖尿病（Type 2 Diabetes, T2D）的发病原理可以简单概括为八个字：**胰岛素抵抗**伴**胰岛素分泌不足**。

这并不是身体完全停止生产胰岛素（那是1型糖尿病），而是身体虽然产生了胰岛素，却无法有效利用它，且胰岛素的分泌最终“跟不上”身体的需求。

以下是详细的发病机制解析，我们可以将其分为三个关键阶段来理解：

### 1. 核心基础：胰岛素抵抗

这是2型糖尿病最初始、最根本的病理改变。

*   **正常情况：** 胰岛素就像一把“钥匙”，人体细胞膜上的受体就像“锁”。当我们进食后血糖升高，胰腺分泌胰岛素，钥匙插入锁孔，打开细胞的大门，血液中的葡萄糖进入细胞内部转化为能量，血糖随之降低。
*   **发病情况：** 在2型糖尿病早期，细胞对胰岛素不敏感了。也就是说，“锁”生锈了或堵塞了，胰岛素这把“钥匙”打不开门。
*   **结果：** 葡萄糖无法顺利进入细胞，堆积在血液里。为了维持血糖稳定，胰腺必须加班加点，分泌比正常人多几倍甚至几十倍的胰岛素来强行“撬开门”。这在医学上称为**“胰岛素抵抗”**。

### 2. 代偿期：高胰岛素血症

在这个阶段，身体虽然出现了胰岛素抵抗，但胰腺功能尚且强大，能够通过过度工作来弥补效率的低下。

*   **表现：** 此时查血糖可能是正常的或仅轻微升高，但血液中的胰岛素水平很高（高胰岛素血症）。
*   **隐患：** 胰腺β细胞（负责生产胰岛素的工厂）长期处于超负荷运转状态，就像一台长期超频运行的发动机，迟早会出现故障。

### 3. 失代偿期：β细胞功能衰竭

这是糖尿病真正确诊的阶段。

*   **疲劳受损：** 长期的超负荷工作导致胰腺β细胞“累垮了”，开始出现凋亡或功能衰退。
*   **分泌不足：** 此时，胰腺分泌胰岛素的能力开始下降，无法再产生足够的胰岛素来克服胰岛素抵抗。
*   **恶性循环：**
    1.  胰岛素不够用 -> 血糖进一步升高。
    2.  高血糖本身对胰腺β细胞有毒性作用（“糖毒性”），进一步抑制β细胞的功能，甚至导致β细胞死亡。
    3.  β细胞

### **步骤 5：参数实验——感受 temperature 的作用**

现在让我们用不同的 temperature 值多次调用同一个问题，观察输出的变化。

In [5]:
# 用不同的 temperature 值调用模型
temperatures = [0.1, 0.5, 1.0]

print("参数实验：Temperature 对输出的影响")
print(f"问题：{medical_question}\n")
print("=" * 80)

for temp in temperatures:
    print(f"\nTemperature = {temp}")
    print("-" * 60)
    
    # 调用 call_llm 函数，传入 temperature 参数
    answer = call_llm(messages, temperature=temp)
    
    print(answer)
    print("-" * 60)

print("\n" + "=" * 80)
print("\n观察与思考：")
print("- Temperature = 0.1：回答是否更稳定、更确定？")
print("- Temperature = 0.7：回答是否有一些变化？")
print("- Temperature = 1.0：回答是否更加多样化？")
print("\n结论：")
print("- 需要准确答案（如医学知识）→ 使用低 temperature（0.1-0.3）")
print("- 需要创意输出（如写作建议）→ 使用高 temperature（0.7-1.0）")

参数实验：Temperature 对输出的影响
问题：请解释糖尿病二型的发病原理。


Temperature = 0.1
------------------------------------------------------------
2型糖尿病（Type 2 Diabetes, T2D）的发病原理可以简单概括为八个字：**“胰岛素抵抗”**加上**“胰岛功能衰竭”**。

为了让你更容易理解，我们可以把身体比作一个**“大型工厂”**，细胞是**“车间”**，葡萄糖（血糖）是**“燃料”**，而胰岛素是**“燃料配送员”**。

以下是详细的发病机制解析：

### 1. 核心机制一：胰岛素抵抗（配送员进不去门）

这是2型糖尿病最早期、最根本的病理改变。

*   **正常情况：** 当你吃饭后，血糖升高，胰腺（工厂总部）派出胰岛素（配送员）去敲细胞（车间）的门。细胞收到信号后打开门，让葡萄糖进入细胞内部燃烧产生能量，血液中的血糖就会降下来。
*   **发病情况：** 在2型糖尿病早期，细胞对胰岛素不敏感了。这就好比配送员去敲门，但车间里的工人听不见，或者门锁生锈了打不开。
*   **结果：** 胰岛素虽然分泌了，但无法有效地让葡萄糖进入细胞。这就叫**“胰岛素抵抗”**。此时，血糖无法被利用，堆积在血液里。

### 2. 核心机制二：代偿期的高胰岛素血症（总部拼命加班）

在胰岛素抵抗发生的初期，身体并不会马上出现高血糖。

*   **身体的反应：** 胰腺（总部）发现：“咦？怎么血糖还没降下来？肯定是配送员不够多！”
*   **代偿机制：** 胰腺开始超负荷工作，分泌比正常人多几倍的胰岛素。这就好比工厂总部为了把燃料送进去，拼命招聘更多的配送员去砸门。
*   **结果：** 此时查血糖可能是正常的，或者稍微偏高，但血液中的胰岛素水平（空腹胰岛素）会显著升高。这被称为**“高胰岛素血症”**，是糖尿病的前奏。

### 3. 核心机制三：胰岛 $\beta$ 细胞功能衰竭（总部累垮了）

这是糖尿病真正确诊的关键转折点。

*   **失代偿：** 胰腺的代偿能力是有限的。长期的高负荷工作，加上长期高血糖对胰腺本身的毒性作用（糖毒性），导致胰腺中的 $\beta$ 细胞（负责制造胰岛素的工人）开始受损、凋亡，甚至“累死”。
*   **分

## **动手实践二：System Prompt 与上下文的影响**

### **目标**

1. 理解 System Prompt 的作用
2. 体验 System Prompt 如何塑造模型的回答风格
3. 理解上下文质量对模型输出的影响
4. 初步认识 RAG（检索增强生成）的概念

---

### **任务 A：System Prompt 的作用**

**什么是 System Prompt？**

System Prompt 是我们在对话开始时，悄悄告诉模型的一条"指令"。它定义了：
- 模型应该扮演什么角色
- 模型应该遵循什么规则
- 模型应该用什么风格回答

**比喻：**
- 就像在演戏前给演员"剧本"和"角色设定"
- System Prompt 就是给大模型的"角色卡"

让我们对比有无 System Prompt 的差异。

In [6]:
# 实验问题
question = "什么是糖尿病？如何进行诊断？"

# 场景 1：没有 System Prompt
print("场景 1：没有 System Prompt")
print("=" * 80)
messages_no_system = [
    {"role": "user", "content": question}
]

answer_no_system = call_llm(messages_no_system, temperature=0.3)
print(answer_no_system)

print("\n" + "=" * 80 + "\n")

# 场景 2：有 System Prompt（扮演医学教授）
print("场景 2：有 System Prompt（扮演医学教授）")
print("=" * 80)

system_prompt = "你是一位耐心细致的医学教授，擅长用通俗易懂的比喻向医学生解释医学概念。你的回答应该清晰、有条理，并且尽量使用生活化的类比。"

messages_with_system = [
    {"role": "system", "content": system_prompt},  # 添加 system prompt
    {"role": "user", "content": question}
]

answer_with_system = call_llm(messages_with_system, temperature=0.3)
print(answer_with_system)

print("\n" + "=" * 80)
print("\n对比观察：")
print("- 场景 1 的回答风格是什么？")
print("- 场景 2 的回答风格有什么不同？")
print("- System Prompt 是否成功塑造了模型的角色？")

场景 1：没有 System Prompt
**糖尿病**是一种以高血糖为特征的代谢性疾病。简单来说，就是你的身体无法正常利用食物中的糖分（葡萄糖），导致血液中的糖分含量过高。

以下是对糖尿病的详细解释及其诊断标准：

### 一、 什么是糖尿病？

当我们进食后，食物中的碳水化合物会转化为葡萄糖进入血液。为了降低血糖，胰腺会分泌一种叫做**胰岛素**的激素，胰岛素就像一把“钥匙”，能打开细胞的大门，让葡萄糖进入细胞内转化为能量。

**糖尿病的发生机制主要有两种：**

1.  **胰岛素分泌不足：** 胰腺这把“锁”坏了，造不出足够的“钥匙”（多见于1型糖尿病）。
2.  **胰岛素作用受阻（胰岛素抵抗）：** 细胞这扇“门”生锈了，有钥匙也打不开（多见于2型糖尿病）。

**主要类型：**
*   **1型糖尿病：** 多发生于儿童和青少年，由于自身免疫反应破坏了胰岛细胞，导致身体几乎不产生胰岛素。患者必须终身注射胰岛素。
*   **2型糖尿病：** 最常见（占90%以上），多发生于中老年人，但近年有年轻化趋势。主要与遗传、肥胖、缺乏运动等生活方式有关。
*   **妊娠期糖尿病：** 在怀孕期间发生，通常产后会恢复，但未来患2型糖尿病的风险增加。

---

### 二、 如何进行诊断？

糖尿病的诊断主要依据血液中的葡萄糖水平。在中国，目前主要采用世界卫生组织（WHO）制定的诊断标准。

**诊断糖尿病的三大“金标准”：**

符合以下任意一条，即可诊断为糖尿病（典型症状见下文）：

1.  **典型糖尿病症状 + 随机血糖 ≥ 11.1 mmol/L**
    *   *典型症状：* “三多一少”（多饮、多尿、多食、体重下降）。
    *   *随机血糖：* 指一天中任意时间的血糖，不考虑上次用餐时间。

2.  **空腹血糖（FPG）≥ 7.0 mmol/L**
    *   *空腹状态：* 至少8小时没有进食热量食物（通常指隔夜空腹，次日早晨抽血）。

3.  **口服葡萄糖耐量试验（OGTT）2小时血糖 ≥ 11.1 mmol/L**
    *   *OGTT：* 患者空腹喝下含有75克葡萄糖的糖水，2小时后测定血糖。

**重要补充：糖化血红蛋白（HbA1c）**
*   近年来，糖化血红蛋白也被纳入诊断标准。如果 HbA1c ≥ 6

### **任务 B：上下文质量的影响**

**什么是上下文（Context）？**

上下文就是我们提供给模型的"背景信息"或"参考材料"。

**比喻：**
- 就像考试时的"开卷允许使用的参考资料"
- 资料质量好，答案就准确
- 资料太多太杂，反而可能找不到重点

让我们体验不同质量的上下文对模型输出的影响。

In [7]:
# 准备上下文材料 - 从文件中加载

# 导入文件读取所需的库
import os

# 上下文 1：长上下文 - 完整的糖尿病防治指南节选
long_context_file = "中国糖尿病防治指南（2024版）- 节选.md"

# 读取长上下文文件
try:
    with open(long_context_file, 'r', encoding='utf-8') as f:
        long_context = f.read()
    print(f"✓ 成功读取长上下文文件：{long_context_file}")
except FileNotFoundError:
    print(f"✗ 文件未找到：{long_context_file}")
    long_context = ""

# 上下文 2：精选上下文1 - 糖尿病诊断相关内容
focused_diagnosis_file = "糖尿病诊断.md"

# 读取糖尿病诊断文件
try:
    with open(focused_diagnosis_file, 'r', encoding='utf-8') as f:
        focused_diagnosis_context = f.read()
    print(f"✓ 成功读取精选上下文文件1：{focused_diagnosis_file}")
except FileNotFoundError:
    print(f"✗ 文件未找到：{focused_diagnosis_file}")
    focused_diagnosis_context = ""

# 上下文 3：精选上下文2 - 中国糖尿病流行的影响因素
focused_factors_file = "中国糖尿病流行的影响因素.md"

# 读取影响因素文件
try:
    with open(focused_factors_file, 'r', encoding='utf-8') as f:
        focused_factors_context = f.read()
    print(f"✓ 成功读取精选上下文文件2：{focused_factors_file}")
except FileNotFoundError:
    print(f"✗ 文件未找到：{focused_factors_file}")
    focused_factors_context = ""

print("\n" + "=" * 80)
print("上下文材料统计：")
print(f"长上下文（完整指南）长度：{len(long_context)} 字符")
print(f"精选上下文1（糖尿病诊断）长度：{len(focused_diagnosis_context)} 字符")
print(f"精选上下文2（流行影响因素）长度：{len(focused_factors_context)} 字符")
print("=" * 80)

✓ 成功读取长上下文文件：中国糖尿病防治指南（2024版）- 节选.md
✓ 成功读取精选上下文文件1：糖尿病诊断.md
✓ 成功读取精选上下文文件2：中国糖尿病流行的影响因素.md

上下文材料统计：
长上下文（完整指南）长度：61269 字符
精选上下文1（糖尿病诊断）长度：2686 字符
精选上下文2（流行影响因素）长度：1690 字符


现在让我们用同一个问题测试这两种不同的上下文。

In [8]:
# 测试问题 - 我们将测试两个不同的问题，分别对应两个精选上下文

# 问题1：关于糖尿病诊断（对应精选上下文1）
test_question_1 = "糖尿病的诊断标准是什么？需要满足哪些条件才能确诊？"

# 问题2：关于糖尿病流行影响因素（对应精选上下文2）
test_question_2 = "中国糖尿病流行的主要影响因素有哪些？"

# ========================================
# 实验1：关于糖尿病诊断的问题
# ========================================
print("=" * 80)
print("实验1：糖尿病诊断问题")
print("=" * 80)
print(f"问题：{test_question_1}\n")

# 场景 1：直接提问，不提供额外上下文
print("场景 1：直接提问，不提供额外上下文")
print("-" * 80)

messages_no_context = [
    {"role": "system", "content": "你是一位医学专家，请回答以下问题。"},
    {"role": "user", "content": f"问题：{test_question_1}"}
]

answer_no_context = call_llm(messages_no_context, temperature=0.1)
print(answer_no_context)

print("\n" + "-" * 80 + "\n")


# 场景 2：使用冗长的上下文（完整指南）- 节选
print("场景 2：使用长上下文（完整指南）- 节选")
print("-" * 80)

messages_long_context = [
    {"role": "system", "content": "你是一位医学专家，请回答以下问题。"},
    {"role": "user", "content": f"请参考以下材料回答问题：\n\n上下文：\n{long_context}\n\n问题：{test_question_1}"}
]

answer_long = call_llm(messages_long_context, temperature=0.1)
print(answer_long)

print("\n" + "-" * 80 + "\n")

# 场景 2：使用精选的上下文（糖尿病诊断）
print("场景 2：使用精选上下文（糖尿病诊断）")
print("-" * 80)

messages_focused_diagnosis = [
    {"role": "system", "content": "你是一位医学专家，请回答以下问题。"},
    {"role": "user", "content": f"请参考以下材料回答问题：\n\n上下文：\n{focused_diagnosis_context}\n\n问题：{test_question_1}"}
]

answer_focused_diagnosis = call_llm(messages_focused_diagnosis, temperature=0.1)
print(answer_focused_diagnosis)

print("\n" + "=" * 80)
print("\n对比观察（问题1）：")
print("- 长上下文答案长度：", len(answer_long))
print("- 精选上下文答案长度：", len(answer_focused_diagnosis))
print("- 场景 1（无上下文）：答案是否准确？")
print("- 场景 2（长上下文）：答案是否准确？有没有被其他信息干扰？和没有上下文对比有什么区别？")
print("- 场景 3（精选上下文）：答案是否更直接、更准确？")

# ========================================
# 实验2：关于糖尿病流行影响因素的问题
# ========================================
print("\n\n" + "=" * 80)
print("实验2：糖尿病流行影响因素问题")
print("=" * 80)
print(f"问题：{test_question_2}\n")

# 场景 1：直接提问，不提供额外上下文
print("场景 1：直接提问，不提供额外上下文")
print("-" * 80)

messages_no_context_2 = [
    {"role": "system", "content": "你是一位医学专家，请回答以下问题。"},
    {"role": "user", "content": f"问题：{test_question_2}"}
]

answer_no_context_2 = call_llm(messages_no_context_2, temperature=0.1)
print(answer_no_context_2)

print("\n" + "-" * 80 + "\n")

# 场景 2：使用冗长的上下文（完整指南）
print("场景 2：使用长上下文（完整指南）")
print("-" * 80)

messages_long_context_2 = [
    {"role": "system", "content": "你是一位医学专家，请根据提供的上下文回答问题。"},
    {"role": "user", "content": f"请参考以下材料回答问题：\n\n上下文：\n{long_context}\n\n问题：{test_question_2}"}
]

answer_long_2 = call_llm(messages_long_context_2, temperature=0.1)
print(answer_long_2)

print("\n" + "-" * 80 + "\n")

# 场景 3：使用精选的上下文（流行影响因素）
print("场景 3：使用精选上下文（流行影响因素）")
print("-" * 80)

messages_focused_factors = [
    {"role": "system", "content": "你是一位医学专家，请根据提供的上下文回答问题。"},
    {"role": "user", "content": f"请参考以下材料回答问题：\n\n上下文：\n{focused_factors_context}\n\n问题：{test_question_2}"}
]

answer_focused_factors = call_llm(messages_focused_factors, temperature=0.1)
print(answer_focused_factors)

print("\n" + "=" * 80)
print("\n对比观察（问题2）：")
print("- 长上下文答案长度：", len(answer_long_2))
print("- 精选上下文答案长度：", len(answer_focused_factors))
print("- 场景 1（无上下文）：答案是否准确？")
print("- 场景 2（长上下文）：答案是否准确？有没有被其他信息干扰？和没有上下文对比有什么区别？")
print("- 场景 3（精选上下文）：答案是否更直接、更准确？")

# ========================================
# 总体对比总结
# ========================================
print("\n\n" + "=" * 80)
print("总体对比总结")
print("=" * 80)
print("\n思考题：")
print("1. 为什么过长的上下文可能导致答案不准或不够简洁？")
print("2. 精选的上下文在哪些方面表现更好？")
print("3. 这对我们设计 RAG（检索增强生成）系统有什么启发？")
print("\n核心概念：")
print("- RAG = 检索增强生成")
print("- 核心思想：先检索最相关的资料，再让模型基于这些精准信息生成答案")
print("- 优势：提高准确性、节省成本、可追溯、可更新")

实验1：糖尿病诊断问题
问题：糖尿病的诊断标准是什么？需要满足哪些条件才能确诊？

场景 1：直接提问，不提供额外上下文
--------------------------------------------------------------------------------
你好。作为医学专家，依据《中国2型糖尿病防治指南（2020年版）》以及世界卫生组织（WHO）的标准，为您详细解答糖尿病的诊断标准。

目前，糖尿病的诊断主要依据静脉血浆葡萄糖水平（血糖）和糖化血红蛋白（HbA1c）水平。

### 一、 糖尿病的诊断标准

满足以下**任意一条**标准，即可诊断为糖尿病：

1.  **典型糖尿病症状 + 随机血糖**
    *   典型症状：包括“三多一少”（多饮、多尿、多食、体重下降）。
    *   随机血糖：指一天中任意时间的血糖，不考虑上次用餐时间。
    *   标准：随机血糖 ≥ 11.1 mmol/L。

2.  **典型糖尿病症状 + 空腹血糖**
    *   空腹状态：指至少8小时没有进食热量。
    *   标准：空腹血糖（FPG） ≥ 7.0 mmol/L。

3.  **典型糖尿病症状 + 葡萄糖负荷后2小时血糖（OGTT）**
    *   OGTT：口服葡萄糖耐量试验。
    *   标准：OGTT 2小时血糖（2h-PG） ≥ 11.1 mmol/L。

4.  **糖化血红蛋白（HbA1c）标准**
    *   标准：HbA1c ≥ 6.5%。
    *   *注意：该指标需要在符合严格质量控制标准的实验室进行检测方可作为诊断依据。*

---

### 二、 确诊需要满足的具体条件

在实际临床操作中，确诊糖尿病并非单纯看一次数值，需结合患者的症状和复查结果，具体逻辑如下：

#### 1. 有典型糖尿病症状
如果患者已经出现了明显的“三多一少”症状，只要**一次**上述检测结果（随机血糖、空腹血糖、OGTT 2h血糖或HbA1c）达到诊断标准，即可确诊为糖尿病。

#### 2. 无典型糖尿病症状
如果患者没有明显症状（这在早期2型糖尿病患者中很常见），则必须满足以下条件之一才能确诊：
*   **改日复查**：第一次检测血糖达到诊断标准，需要在另一天复查，如果复查结果依然达到上述诊断标准，方可

### **引入概念：RAG（检索增强生成）**

**什么是 RAG？**

RAG = Retrieval-Augmented Generation（检索增强生成）

**核心思想：**

与其把一大堆资料全都扔给模型（像场景 1 那样），不如：
1. **先检索**：根据用户的问题，从海量资料中找到最相关的一小部分
2. **再生成**：只把这部分相关资料喂给模型，让它基于这些精准信息生成答案

**比喻：**

- **传统方式（场景 1）**：把整本医学词典给学生，让他自己找答案
- **RAG 方式**：老师先帮学生翻到最相关的几页，再让他基于这几页回答

**为什么需要 RAG？**

1. **提高准确性**：减少无关信息干扰
2. **节省成本**：减少 Token 使用量（API 按字数收费）
3. **可追溯性**：知道答案来自哪份资料
4. **可更新性**：资料更新时，只需更新检索库，不用重新训练模型

**RAG 的基本流程：**
```
用户问题
    ↓
【检索模块】→ 在知识库中找到最相关的文档片段
    ↓
【增强模块】→ 将问题 + 相关片段 组合成 Prompt
    ↓
【生成模块】→ LLM 基于 Prompt 生成答案
    ↓
最终答案
```

**RAG 的应用场景：**
- 医学知识问答系统
- 企业文档智能检索
- 临床决策支持系统
- 法律文件分析

**这是进阶内容**：

RAG 涉及向量数据库、语义检索等更复杂的技术。本节课我们只建立基本概念，如果你感兴趣，可以后续深入学习！

## **动手实践三：病例文本结构化**

### **目标**

通过一个实用的医学案例，学习如何用大模型从自由文本中提取结构化信息。

**场景描述：**

你是医院信息科的一名工程师。医院积累了大量的电子病历，但都是自由文本形式。医生希望将这些病历信息提取出来，整理成结构化的表格，方便后续的数据分析和研究。

**我们的任务：**
1. 设计一个 Prompt，让模型从病例文本中提取关键信息
2. 处理单个病例，验证方法有效
3. 批量处理多个病例
4. 将结果保存为 CSV 文件

---

### **步骤 1：单一样例演示**

首先，让我们用一个简单的病例来测试我们的方法。

In [9]:
# 示例病例文本
sample_case = """
患者张先生，56岁，于2024年3月15日因"多饮多尿2个月，加重1周"来院就诊。
患者2个月前无明显诱因出现多饮、多尿症状，每日饮水量约3000ml，尿量相应增多。
1周前症状加重，伴有乏力、体重下降约5kg。无发热、无腹痛、无意识障碍。
既往史：高血压病史5年，规律服用氨氯地平5mg qd，血压控制尚可。
个人史：吸烟史30年，每日10支；偶有饮酒。
家族史：父亲有糖尿病史。

体格检查：T 36.5℃，P 88次/分，R 18次/分，BP 135/85mmHg，
BMI 26.5kg/m²。神志清楚，精神可。心肺听诊无异常。腹软，无压痛。

辅助检查：
空腹血糖：8.6 mmol/L（正常值：3.9-6.1）
餐后2小时血糖：14.2 mmol/L（正常值：<7.8）
糖化血红蛋白：7.8%（正常值：4-6）
尿常规：尿糖(++)，尿蛋白(-)

初步诊断：2型糖尿病
"""

print("示例病例：")
print("=" * 80)
print(sample_case)
print("=" * 80)

示例病例：

患者张先生，56岁，于2024年3月15日因"多饮多尿2个月，加重1周"来院就诊。
患者2个月前无明显诱因出现多饮、多尿症状，每日饮水量约3000ml，尿量相应增多。
1周前症状加重，伴有乏力、体重下降约5kg。无发热、无腹痛、无意识障碍。
既往史：高血压病史5年，规律服用氨氯地平5mg qd，血压控制尚可。
个人史：吸烟史30年，每日10支；偶有饮酒。
家族史：父亲有糖尿病史。

体格检查：T 36.5℃，P 88次/分，R 18次/分，BP 135/85mmHg，
BMI 26.5kg/m²。神志清楚，精神可。心肺听诊无异常。腹软，无压痛。

辅助检查：
空腹血糖：8.6 mmol/L（正常值：3.9-6.1）
餐后2小时血糖：14.2 mmol/L（正常值：<7.8）
糖化血红蛋白：7.8%（正常值：4-6）
尿常规：尿糖(++)，尿蛋白(-)

初步诊断：2型糖尿病



现在，让我们设计一个 System Prompt，指示模型如何从这段文本中提取信息。

In [10]:
# 设计信息提取的 System Prompt
extraction_system_prompt = """
你是一位专业的医学信息提取专家。你的任务是从病历文本中提取关键信息，并以 JSON 格式输出。

请提取以下字段（如果文本中没有某个字段的信息，请填写"未提及"）：

1. patient_name: 患者姓名
2. age: 患者年龄
3. gender: 患者性别
4. chief_complaint: 主诉（患者的主要症状和就诊原因）
5. present_illness: 现病史（当前疾病的详细描述）
6. past_history: 既往史（患者过去的疾病史）
7. family_history: 家族史
8. vital_signs: 重要体征（体温、脉搏、呼吸、血压等）
9. lab_results: 关键实验室检查结果
10. diagnosis: 初步诊断

输出格式要求：
- 只输出纯粹的 JSON 格式，不要有任何额外文字
- 不要使用代码块标记（如 ```json），直接以 { 开头，以 } 结尾
- 所有字段名使用英文，如上所示
- 字段值使用中文，保持简洁
"""

# TODO: 构建包含 system prompt 的消息
# 提示：role 应该是 "system" 和 "user"
extraction_messages = [
    {"role": "system", "content": extraction_system_prompt},
    {"role": "user", "content": f"请从以下病历中提取信息：\n\n{sample_case}"}
]

print("消息构建完成！")

消息构建完成！


现在调用模型，提取信息。

In [11]:
# 调用模型进行信息提取
print("正在提取病例信息...")
print("=" * 80)

# 调用 call_llm 函数，使用较低的 temperature（如 0.1）以确保输出稳定
extraction_result = call_llm(extraction_messages, temperature=0.1)

print(extraction_result)
print("=" * 80)

# 解析 JSON
try:
    # 使用 json.loads 解析提取结果
    extracted_data = json.loads(extraction_result)
    print("\nJSON 解析成功！")
    print("\n提取的信息：")
    for key, value in extracted_data.items():
        print(f"  {key}: {value}")
except json.JSONDecodeError as e:
    print(f"\nJSON 解析失败：{e}")
    print("\n可能的原因：")
    print("- 模型输出不是有效的 JSON 格式")
    print("- 输出中包含了额外的文字说明")
    extracted_data = None

正在提取病例信息...
{
"patient_name": "张先生",
"age": "56岁",
"gender": "男",
"chief_complaint": "多饮多尿2个月，加重1周",
"present_illness": "患者2个月前无明显诱因出现多饮、多尿症状，每日饮水量约3000ml，尿量相应增多。1周前症状加重，伴有乏力、体重下降约5kg。无发热、无腹痛、无意识障碍。",
"past_history": "高血压病史5年，规律服用氨氯地平5mg qd，血压控制尚可。",
"family_history": "父亲有糖尿病史。",
"vital_signs": "T 36.5℃，P 88次/分，R 18次/分，BP 135/85mmHg，BMI 26.5kg/m²",
"lab_results": "空腹血糖8.6 mmol/L，餐后2小时血糖14.2 mmol/L，糖化血红蛋白7.8%，尿糖(++)，尿蛋白(-)",
"diagnosis": "2型糖尿病"
}

JSON 解析成功！

提取的信息：
  patient_name: 张先生
  age: 56岁
  gender: 男
  chief_complaint: 多饮多尿2个月，加重1周
  present_illness: 患者2个月前无明显诱因出现多饮、多尿症状，每日饮水量约3000ml，尿量相应增多。1周前症状加重，伴有乏力、体重下降约5kg。无发热、无腹痛、无意识障碍。
  past_history: 高血压病史5年，规律服用氨氯地平5mg qd，血压控制尚可。
  family_history: 父亲有糖尿病史。
  vital_signs: T 36.5℃，P 88次/分，R 18次/分，BP 135/85mmHg，BMI 26.5kg/m²
  lab_results: 空腹血糖8.6 mmol/L，餐后2小时血糖14.2 mmol/L，糖化血红蛋白7.8%，尿糖(++)，尿蛋白(-)
  diagnosis: 2型糖尿病


### **步骤 2：批量处理与保存**

单个病例成功了！现在让我们处理多个病例，并将结果保存为 CSV 文件。

首先，准备多个病例文本。

In [12]:
# 准备多个病例文本
cases = [
    """
    患者李女士，45岁，因"反复心悸、胸闷3个月"就诊。
    患者3个月前在情绪激动后出现心悸、胸闷，每次持续约10分钟，休息后可缓解。
    近1周发作频率增加，每日2-3次。无胸痛、无晕厥。
    既往史：体健。无高血压、糖尿病史。
    个人史：不吸烟、不饮酒。工作压力较大。
    家族史：母亲有冠心病史。
    
    体格检查：T 36.8℃，P 92次/分，R 18次/分，BP 120/80mmHg。
    神志清楚，精神可。心律不齐，可闻及早搏，心音正常。双肺呼吸音清。
    
    辅助检查：
    心电图：窦性心律，频发室性早搏。
    心脏彩超：结构正常，左室射血分数 60%。
    
    初步诊断：心律失常（室性早搏）
    """,
    
    """
    患者王大爷，68岁，因"咳嗽、咳痰1周，发热2天"来院。
    患者1周前受凉后出现咳嗽、咳白色黏痰，量不多。2天前出现发热，
    体温最高38.5℃，伴畏寒、乏力。无胸痛、无呼吸困难。
    既往史：慢性阻塞性肺疾病（COPD）病史10年，长期使用吸入剂。
    高血压病史8年，规律服药。
    
    体格检查：T 38.2℃，P 96次/分，R 24次/分，BP 140/90mmHg，SpO2 92%。
    神志清楚，精神稍差。口唇轻度发绀。双肺呼吸音粗，可闻及湿啰音。
    心律齐，心音正常。
    
    辅助检查：
    血常规：WBC 12.5×10^9/L，N 85%
    胸部CT：右下肺可见斑片状阴影。
    
    初步诊断：社区获得性肺炎
    """,
    
    """
    患者小陈，28岁，因"上腹痛3天，加重1天"就诊。
    患者3天前因暴饮暴食后出现上腹部隐痛，呈持续性，与进食无关。
    1天前疼痛加重，伴恶心、呕吐2次，为胃内容物。无发热、无黑便。
    既往史：体健。无慢性病史。
    个人史：经常熬夜，饮食不规律。
    
    体格检查：T 36.6℃，P 80次/分，R 18次/分，BP 110/70mmHg。
    神志清楚，精神可。腹软，上腹部压痛(+)，无反跳痛，
    肝脾肋下未触及。肠鸣音正常。
    
    辅助检查：
    血常规：WBC 9.8×10^9/L，N 72%
    血淀粉酶：正常
    腹部B超：肝胆胰脾未见明显异常。
    
    初步诊断：急性胃炎
    """,
    
    """
    患者赵女士，52岁，因"发现乳房肿块1个月"就诊。
    患者1个月前自查发现右乳外上象限有一约2cm×2cm肿块，
    无疼痛，无乳头溢液，无皮肤改变。肿块大小无明显变化。
    既往史：体健。月经规律，已绝经1年。
    家族史：阿姨曾患乳腺癌。
    
    体格检查：T 36.5℃，P 76次/分，R 18次/分，BP 120/75mmHg。
    神志清楚，精神可。右乳外上象限可触及约2cm×2cm肿块，
    质地硬，边界不清，活动度差，无压痛。腋窝淋巴结未触及肿大。
    
    辅助检查：
    乳腺超声：右乳外上象限低回声结节，BI-RADS 4C类。
    钼靶：右乳外上象限不规则肿块，伴有细小钙化。
    
    初步诊断：右乳肿块（性质待查，疑似恶性肿瘤）
    """,
    
    """
    患者刘先生，62岁，因"排尿困难、尿频半年，加重1个月"就诊。
    患者半年前开始出现排尿困难，尿线变细，尿频、夜尿增多（每晚3-4次）。
    1个月前症状加重，伴尿急、尿痛。无血尿，无腰痛。
    既往史：高血压病史5年，控制良好。
    
    体格检查：T 36.7℃，P 78次/分，R 16次/分，BP 130/80mmHg。
    神志清楚，精神可。心肺听诊无异常。腹软，无压痛。
    直肠指检：前列腺Ⅱ度增大，质地中等，表面光滑，无结节，中央沟变浅。
    
    辅助检查：
    尿常规：WBC(+)，RBC(-)
    前列腺特异性抗原（PSA）：4.2 ng/ml（正常值：<4）
    前列腺超声：前列腺增大，约 5.2cm×4.1cm×3.8cm，内部回声均匀。
    
    初步诊断：良性前列腺增生
    """
]

print(f"准备了 {len(cases)} 个病例")
print("\n病例列表：")
for i, case in enumerate(cases, 1):
    # 提取第一行作为简要描述
    first_line = case.strip().split('\n')[0]
    print(f"  {i}. {first_line}")

准备了 5 个病例

病例列表：
  1. 患者李女士，45岁，因"反复心悸、胸闷3个月"就诊。
  2. 患者王大爷，68岁，因"咳嗽、咳痰1周，发热2天"来院。
  3. 患者小陈，28岁，因"上腹痛3天，加重1天"就诊。
  4. 患者赵女士，52岁，因"发现乳房肿块1个月"就诊。
  5. 患者刘先生，62岁，因"排尿困难、尿频半年，加重1个月"就诊。


现在，让我们编写批量处理代码。

In [13]:
# 批量提取病例信息

print("开始批量处理病例...")
print("=" * 80)

# TODO: 创建一个列表来存储所有提取结果
# 提示：使用空列表 []
all_extracted_data = []

# 遍历每个病例
for i, case in enumerate(cases, 1):
    print(f"\n正在处理第 {i}/{len(cases)} 个病例...")
    
    # 构建消息
    messages = [
        {"role": "system", "content": extraction_system_prompt},
        {"role": "user", "content": f"请从以下病历中提取信息：\n\n{case}"}
    ]
    
    # 调用模型
    result = call_llm(messages, temperature=0.1)
    
    # 解析 JSON
    try:
        data = json.loads(result)
        # 添加病例编号
        data['case_id'] = i
        all_extracted_data.append(data)
        print(f"  第 {i} 个病例提取成功")
    except json.JSONDecodeError as e:
        print(f"  第 {i} 个病例提取失败：{e}")
        print(f"     模型输出：{result[:100]}...")

print("\n" + "=" * 80)
print(f"\n批量处理完成！成功提取 {len(all_extracted_data)}/{len(cases)} 个病例")

开始批量处理病例...

正在处理第 1/5 个病例...
  第 1 个病例提取成功

正在处理第 2/5 个病例...
  第 2 个病例提取成功

正在处理第 3/5 个病例...
  第 3 个病例提取成功

正在处理第 4/5 个病例...
  第 4 个病例提取成功

正在处理第 5/5 个病例...
  第 5 个病例提取成功


批量处理完成！成功提取 5/5 个病例


接下来，让我们将结果转换为 DataFrame 并保存为 CSV 文件。

In [14]:
# 导入 pandas
import pandas as pd

print("pandas 库准备完成！")

pandas 库准备完成！


In [15]:
# 将提取的数据转换为 DataFrame
df = pd.DataFrame(all_extracted_data)

# 调整列的顺序（将 case_id 放在最前面）
if 'case_id' in df.columns:
    cols = ['case_id'] + [col for col in df.columns if col != 'case_id']
    df = df[cols]

# 显示 DataFrame
print("提取结果预览：")
print("=" * 80)
print(df.to_string())
print("=" * 80)

# 显示 DataFrame 信息
print(f"\nDataFrame 信息：")
print(f"  - 行数（病例数）：{len(df)}")
print(f"  - 列数（字段数）：{len(df.columns)}")
print(f"  - 列名：{list(df.columns)}")

提取结果预览：
   case_id patient_name  age gender  chief_complaint                                              present_illness                               past_history family_history                                     vital_signs                                                                       lab_results          diagnosis
0        1          李女士  45岁      女       反复心悸、胸闷3个月  3个月前情绪激动后出现心悸、胸闷，每次持续约10分钟，休息后可缓解。近1周发作频率增加，每日2-3次。无胸痛、无晕厥。                              体健，无高血压、糖尿病史。       母亲有冠心病史。           T 36.8℃，P 92次/分，R 18次/分，BP 120/80mmHg                                             心电图：窦性心律，频发室性早搏；心脏彩超：结构正常，左室射血分数 60%。         心律失常（室性早搏）
1        2          王大爷  68岁      男     咳嗽、咳痰1周，发热2天     1周前受凉后出现咳嗽、咳白色黏痰，量不多。2天前出现发热，体温最高38.5℃，伴畏寒、乏力。无胸痛、无呼吸困难。  慢性阻塞性肺疾病（COPD）病史10年，长期使用吸入剂；高血压病史8年，规律服药。            未提及  T 38.2℃，P 96次/分，R 24次/分，BP 140/90mmHg，SpO2 92%                                        血常规：WBC 12.5×10^9/L，N 85%；胸部CT：右下肺可见斑片状阴影。            社区获得性肺炎
2        3           小陈  28岁 

最后，将 DataFrame 保存为 CSV 文件。

In [16]:
# 将 DataFrame 保存为 CSV 文件
output_file = "病例结构化结果.csv"

# 保存为 CSV，不保存索引列
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"文件已保存：{output_file}")
print(f"文件位置：{os.path.abspath(output_file)}")

# 读取并显示保存的文件内容（验证）
print("\n文件内容预览：")
print("=" * 80)
with open(output_file, 'r', encoding='utf-8-sig') as f:
    # 显示前 500 个字符
    content = f.read()
    print(content[:500] + "..." if len(content) > 500 else content)
print("=" * 80)

文件已保存：病例结构化结果.csv
文件位置：D:\Jupyter notebook project\AI for medicine new\week 8\病例结构化结果.csv

文件内容预览：
case_id,patient_name,age,gender,chief_complaint,present_illness,past_history,family_history,vital_signs,lab_results,diagnosis
1,李女士,45岁,女,反复心悸、胸闷3个月,3个月前情绪激动后出现心悸、胸闷，每次持续约10分钟，休息后可缓解。近1周发作频率增加，每日2-3次。无胸痛、无晕厥。,体健，无高血压、糖尿病史。,母亲有冠心病史。,T 36.8℃，P 92次/分，R 18次/分，BP 120/80mmHg,心电图：窦性心律，频发室性早搏；心脏彩超：结构正常，左室射血分数 60%。,心律失常（室性早搏）
2,王大爷,68岁,男,咳嗽、咳痰1周，发热2天,1周前受凉后出现咳嗽、咳白色黏痰，量不多。2天前出现发热，体温最高38.5℃，伴畏寒、乏力。无胸痛、无呼吸困难。,慢性阻塞性肺疾病（COPD）病史10年，长期使用吸入剂；高血压病史8年，规律服药。,未提及,T 38.2℃，P 96次/分，R 24次/分，BP 140/90mmHg，SpO2 92%,血常规：WB...


### **实践三总结**

我们成功完成了以下任务：

1. 设计了信息提取的 System Prompt
2. 成功从单个病例中提取结构化信息
3. 批量处理了多个病例
4. 将结果转换为 DataFrame 并保存为 CSV 文件

**这个案例的实用性：**

- 将杂乱的病历文本转化为结构化数据
- 便于后续的数据分析和研究
- 可以扩展到其他类型的文本信息提取
- 是医疗信息化的重要应用方向

**可能的改进方向：**

- 增加数据验证（如年龄必须是数字）
- 处理更复杂的病例格式
- 添加错误处理和重试机制
- 使用更快的模型以提高批量处理效率

## **本节课总结**

### **我们学到了什么？**

1. **大模型基本原理**
   - Tokenizer：把文本切碎成 Token
   - Embedding：把 Token 转换成有意义的数字坐标
   - LLM 核心：强大的下一个词预测器

2. **API 调用基础**
   - API 和 API Key 的概念
   - 如何获取和使用 API Key
   - 常见参数（temperature、top_p、max_tokens）的作用

3. **Prompt 工程入门**
   - System Prompt 的作用：塑造模型的角色和风格
   - 上下文质量对模型输出的影响
   - RAG 概念：检索增强生成

4. **实用技能**
   - 使用 Python 调用大模型 API
   - 从文本中提取结构化信息
   - 批量处理并保存结果

---

### **与医学的关系**

大模型在医学领域有广泛的应用前景：

- 病历结构化：将自由文本病历转化为结构化数据（本节课案例）
- 辅助诊断：基于患者信息提供诊断建议
- 医学知识问答：快速回答医学问题
- 病历书写辅助：帮助医生快速生成病历初稿
- 医患沟通：生成患者教育材料
- 临床研究：从海量文献中提取关键信息

---

### **思考与扩展**

1. **思考题**
   - 如果你要从医学文献中提取研究对象、方法、结果、结论，应该如何设计 Prompt？
   - 在什么情况下应该使用高 temperature？什么情况下应该使用低 temperature？
   - 为什么有些时候模型会胡说八道（产生幻觉）？如何减少这种情况？

2. **扩展练习**
   - 尝试处理你自己收集的病例文本
   - 修改 System Prompt，提取不同的字段（如用药信息、检查项目等）
   - 尝试用不同的模型（如 Qwen-3.6-PLUS）对比输出结果

3. **进阶学习方向**
   - 深入学习 Prompt Engineering（提示词工程）
   - 学习 RAG 技术的实现
   - 探索大模型的微调（Fine-tuning）方法
   - 了解医疗领域的大模型应用伦理和安全问题


---

**恭喜你完成了本节课的学习！**

现在你已经掌握了大模型的基础知识和在医学中的初步应用方法。
继续探索，你将发现大模型在医疗领域的无限可能！